Notebook: 09_confidence_features.ipynb

Purpose: Assemble record-level confidence evidence from all permitted domains.

Inputs:
- signal_quality_features.parquet
- delineation_features.parquet
- twave_features.parquet
- measurement_reliability.parquet
- clinical_context.parquet

Outputs:
- confidence_features.parquet

# 09 — Confidence Feature Assembly

Aggregate lead- and beat-level evidence into the master record-level feature table for confidence modeling.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd().parent / 'src'))

root_dir = Path.cwd().parent
artifacts_dir = root_dir / 'artifacts'
frames = {
    'signal_quality': pd.read_parquet(artifacts_dir / 'signal_quality_features.parquet') if (artifacts_dir / 'signal_quality_features.parquet').exists() else pd.DataFrame(),
    'delineation': pd.read_parquet(artifacts_dir / 'delineation_features.parquet') if (artifacts_dir / 'delineation_features.parquet').exists() else pd.DataFrame(),
    'twave': pd.read_parquet(artifacts_dir / 'twave_features.parquet') if (artifacts_dir / 'twave_features.parquet').exists() else pd.DataFrame(),
    'measurement': pd.read_parquet(artifacts_dir / 'measurement_reliability.parquet') if (artifacts_dir / 'measurement_reliability.parquet').exists() else pd.DataFrame(),
    'clinical': pd.read_parquet(artifacts_dir / 'clinical_context.parquet') if (artifacts_dir / 'clinical_context.parquet').exists() else pd.DataFrame(),
}


def aggregations(series):
    if series.empty:
        return dict(mean=np.nan, std=np.nan, max=np.nan, p95=np.nan)
    values = pd.to_numeric(series.dropna(), errors='coerce')
    if values.empty:
        return dict(mean=np.nan, std=np.nan, max=np.nan, p95=np.nan)
    return dict(mean=float(values.mean()), std=float(values.std()), max=float(values.max()), p95=float(np.nanpercentile(values, 95)))

all_record_ids = set()
for df in frames.values():
    if 'record_id' in df.columns:
        all_record_ids.update(df['record_id'].astype(str).unique())

rows = []
for record_id in sorted(all_record_ids):
    row = {'record_id': record_id}
    clinical = frames['clinical']
    if not clinical.empty:
        rec = clinical[clinical['record_id'] == record_id]
        if not rec.empty:
            row['arrhythmia_flag'] = bool(rec['arrhythmia_flag'].iat[0])
            row['diagnostic_class'] = rec['diagnostic_class'].iat[0]
            row['dataset_origin'] = rec['dataset_origin'].iat[0]
        else:
            row['arrhythmia_flag'] = False
            row['diagnostic_class'] = np.nan
            row['dataset_origin'] = record_id.split('/', 1)[0] if '/' in record_id else 'unknown'
    else:
        row['arrhythmia_flag'] = False
        row['diagnostic_class'] = np.nan
        row['dataset_origin'] = record_id.split('/', 1)[0] if '/' in record_id else 'unknown'

    sq = frames['signal_quality']
    de = frames['delineation']
    tw = frames['twave']
    mr = frames['measurement']

    for col in ['hfn_index','pli_index','clipping_ratio','flatline_ratio','signal_quality_score']:
        agg = aggregations(sq[sq['record_id'] == record_id][col]) if not sq.empty else dict(mean=np.nan, std=np.nan, max=np.nan, p95=np.nan)
        row[f'mean_{col}'] = agg['mean']
        row[f'std_{col}'] = agg['std']
        row[f'max_{col}'] = agg['max']
        row[f'p95_{col}'] = agg['p95']

    boundary_conf = aggregations(de[de['record_id'] == record_id]['boundary_confidence']) if not de.empty else {'mean': np.nan, 'std': np.nan, 'max': np.nan, 'p95': np.nan}
    row['mean_boundary_confidence'] = boundary_conf['mean']
    row['std_boundary_confidence'] = boundary_conf['std']
    row['max_t_end_uncertainty_ms'] = aggregations(de[de['record_id'] == record_id]['t_end_uncertainty_ms'])['max'] if not de.empty else np.nan
    row['p95_t_end_uncertainty_ms'] = aggregations(de[de['record_id'] == record_id]['t_end_uncertainty_ms'])['p95'] if not de.empty else np.nan

    row['mean_t_end_ambiguity_score'] = aggregations(tw[tw['record_id'] == record_id]['t_end_ambiguity_score'])['mean'] if not tw.empty else np.nan
    row['max_t_end_ambiguity_score'] = aggregations(tw[tw['record_id'] == record_id]['t_end_ambiguity_score'])['max'] if not tw.empty else np.nan
    row['mean_morphology_confidence'] = aggregations(tw[tw['record_id'] == record_id]['morphology_confidence'])['mean'] if not tw.empty else np.nan

    row['mean_bsqi'] = aggregations(mr[mr['record_id'] == record_id]['bsqi'])['mean'] if not mr.empty else np.nan
    row['min_bsqi'] = aggregations(mr[mr['record_id'] == record_id]['bsqi'])['mean'] if not mr.empty else np.nan
    row['mean_wsqi'] = aggregations(mr[mr['record_id'] == record_id]['wsqi'])['mean'] if not mr.empty else np.nan
    row['min_wsqi'] = aggregations(mr[mr['record_id'] == record_id]['wsqi'])['mean'] if not mr.empty else np.nan
    row['mean_lead_agreement'] = aggregations(mr[mr['record_id'] == record_id]['lead_agreement_score'])['mean'] if not mr.empty else np.nan
    row['worst_lead_agreement'] = aggregations(mr[mr['record_id'] == record_id]['lead_agreement_score'])['max'] if not mr.empty else np.nan
    row['mean_beat_agreement'] = aggregations(mr[mr['record_id'] == record_id]['beat_agreement_score'])['mean'] if not mr.empty else np.nan
    row['worst_beat_agreement'] = aggregations(mr[mr['record_id'] == record_id]['beat_agreement_score'])['max'] if not mr.empty else np.nan
    row['qt_variance_leads'] = aggregations(mr[mr['record_id'] == record_id]['qt_variance_leads'])['mean'] if not mr.empty else np.nan
    row['qt_variance_beats'] = aggregations(mr[mr['record_id'] == record_id]['qt_variance_beats'])['mean'] if not mr.empty else np.nan

    rows.append(row)

confidence_features = pd.DataFrame(rows)
required = {
    'record_id','arrhythmia_flag','diagnostic_class','dataset_origin',
    'mean_hfn_index','std_hfn_index','max_hfn_index','p95_hfn_index',
    'mean_pli_index','std_pli_index','max_pli_index','p95_pli_index',
    'mean_clipping_ratio','std_clipping_ratio','max_clipping_ratio','p95_clipping_ratio',
    'mean_flatline_ratio','std_flatline_ratio','max_flatline_ratio','p95_flatline_ratio',
    'mean_signal_quality_score','std_signal_quality_score','max_signal_quality_score','p95_signal_quality_score',
    'mean_boundary_confidence','std_boundary_confidence','max_t_end_uncertainty_ms','p95_t_end_uncertainty_ms',
    'mean_t_end_ambiguity_score','max_t_end_ambiguity_score','mean_morphology_confidence',
    'mean_bsqi','min_bsqi','mean_wsqi','min_wsqi','mean_lead_agreement','worst_lead_agreement','mean_beat_agreement','worst_beat_agreement',
    'qt_variance_leads','qt_variance_beats',
}
missing = required.difference(set(confidence_features.columns))
assert not missing, f'missing required columns: {sorted(missing)}'
assert confidence_features['record_id'].is_unique

confidence_features.to_parquet(artifacts_dir / 'confidence_features.parquet', index=False)
print('Wrote confidence_features.parquet')
